<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3 Transfer with Cosmos Framework

This notebook runs Cosmos3 **video transfer** inference through the native Cosmos Framework PyTorch entrypoint:

```bash
python -m cosmos_framework.scripts.inference
```

Transfer generates a target clip from a caption (`prompt.json`) and a spatial control video on the hint block (`control_path`). Supported cookbook controls:

- **edge** — Canny edge map (`control_edge.mp4`)
- **blur** — blurred reference (`control_blur.mp4`)
- **depth** — depth map (`control_depth.mp4`)
- **seg** — segmentation map (`control_seg.mp4`)
- **wsm** — world-scenario map (`control_wsm.mp4`)

vLLM-Omni does not expose transfer controls today; use this Cosmos Framework path only.

Set **`COSMOS3_MODEL`** in §2 to choose the model before running §9–§13:

| `COSMOS3_MODEL` | Launcher | Parallelism |
|---|---|---|
| `Cosmos3-Nano` (default) | `python` (single GPU) | `latency` |
| `Cosmos3-Super` | `torchrun` (multi-GPU) | `throughput` |

> **GPU required.** Run on a host where §3 (`nvidia-smi`) and §7 (`cuda available: True`) both pass.

**Self-contained setup:** everything needed to run this notebook (system packages, clone, Python venv) is in §2–§7 below — no external bootstrap scripts required.

Workflow: §2 configure → §3 GPU check → §4 system packages → §5 clone framework → §6 install → §7 verify → §8 review specs → §9–§13 inference and preview (run only the controls you need).


## 1. Prerequisites

1. Linux machine with an NVIDIA GPU.
2. A [Hugging Face account](https://huggingface.co) with access to the Cosmos3 model repos — paste your token into **§2 below**.
3. `git` available on PATH (§5 clones Cosmos Framework when missing).
4. Outbound internet for `git clone`, PyPI (`uv sync` in §6), and HF checkpoint downloads.

Everything else — system packages, framework clone, Python venv — is installed automatically by §4–§6.


## 2. Configure

**Edit the cell directly below** — it is the only cell you need to change before running the notebook top to bottom.

| Setting | What it controls |
|---|---|
| `HF_TOKEN` | Hugging Face token for downloading model weights |
| `COSMOS3_CACHE_ROOT` | Path for uv + HF caches (leave `""` to use default under this cookbook) |
| `COSMOS3_TRANSFER_OUTPUT_ROOT` | Where generated videos are saved (leave `""` for default) |

Inference cells (§9–§13) default to **Cosmos3-Nano**. To run **Cosmos3-Super**, add `os.environ["COSMOS3_MODEL"] = "Cosmos3-Super"` to the config cell below.


In [ ]:
# ── Edit here, then run all cells ──────────────────────────────────────────

# Hugging Face token — required to download Cosmos3 weights.
# Get yours at https://huggingface.co/settings/tokens
HF_TOKEN = ""  # e.g. "hf_xxxxxxxxxxxxxxxxxxxxxxxx"

# Cache root for uv and Hugging Face downloads.
# Set to a large disk, e.g. "/lustre/scratch/cache". Leave "" for default.
COSMOS3_CACHE_ROOT = ""

# Output directory for generated videos. Leave "" for default.
COSMOS3_TRANSFER_OUTPUT_ROOT = ""

# ── Push to environment (do not edit below this line) ───────────────────────
import os
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
if COSMOS3_CACHE_ROOT:
    os.environ["COSMOS3_CACHE_ROOT"] = COSMOS3_CACHE_ROOT
if COSMOS3_TRANSFER_OUTPUT_ROOT:
    os.environ["COSMOS3_TRANSFER_OUTPUT_ROOT"] = COSMOS3_TRANSFER_OUTPUT_ROOT
print(f"cache: {COSMOS3_CACHE_ROOT or '(default)'}")
print(f"HF_TOKEN: {'set' if HF_TOKEN else 'not set — using existing hf login session'}")


## 3. Confirm GPU access

Run this **before** install (§6) or inference (§9+). If it fails, fix GPU allocation or driver setup before continuing.

In [1]:
from pathlib import Path
import json
import os
import platform
import socket


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


def free_local_port() -> str:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return str(sock.getsockname()[1])


def default_framework_repo(root: Path) -> Path:
    for candidate in (root / "packages" / "cosmos-framework", root / "packages" / "cosmos3"):
        if (candidate / "pyproject.toml").exists() and (candidate / "cosmos_framework").exists():
            return candidate
    return root / "packages" / "cosmos3"


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
COSMOS3_TRANSFER_ROOT = COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "transfer"
COSMOS3_REPO = Path(os.environ.get("COSMOS3_REPO", default_framework_repo(COSMOS_ROOT))).resolve()
COSMOS3_GIT_URL = os.environ.get(
    "COSMOS3_GIT_URL",
    "https://github.com/NVIDIA/cosmos-framework.git",
)
def default_uv_group() -> str:
    # cu128-train attention wheels are x86_64-only; aarch64 hosts need cu130-train for natten.
    if platform.machine() == "aarch64":
        return "cu130-train"
    return "cu128-train"


COSMOS3_UV_GROUP = os.environ.get("COSMOS3_UV_GROUP", default_uv_group())
COSMOS3_TRANSFER_OUTPUT_ROOT = Path(
    os.environ.get(
        "COSMOS3_TRANSFER_OUTPUT_ROOT",
        COSMOS3_TRANSFER_ROOT / "outputs" / "notebooks",
    )
).resolve()
COSMOS3_SPECS_DIR = COSMOS3_TRANSFER_ROOT / "specs"
TRANSFER_CONTROLS = ("edge", "blur", "depth", "seg", "wsm")
COSMOS3_MODEL = os.environ.get("COSMOS3_MODEL", "Cosmos3-Nano")


def _detect_gpu_count() -> str:
    """Count visible GPUs via nvidia-smi; fall back to 4."""
    import subprocess
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            timeout=10, stderr=subprocess.DEVNULL,
        ).decode()
        count = len([l for l in out.strip().splitlines() if l])
        if count > 0:
            return str(count)
    except Exception:
        pass
    return "4"


COSMOS3_NUM_GPUS = os.environ.get("COSMOS3_NUM_GPUS") or _detect_gpu_count()

_VALID_MODELS = ("Cosmos3-Nano", "Cosmos3-Super")
if COSMOS3_MODEL not in _VALID_MODELS:
    raise ValueError(f"COSMOS3_MODEL must be one of {_VALID_MODELS}, got {COSMOS3_MODEL!r}")

os.environ["COSMOS_ROOT"] = str(COSMOS_ROOT)
os.environ["COSMOS3_TRANSFER_ROOT"] = str(COSMOS3_TRANSFER_ROOT)
os.environ["COSMOS3_REPO"] = str(COSMOS3_REPO)
os.environ["COSMOS3_GIT_URL"] = COSMOS3_GIT_URL
os.environ["COSMOS3_UV_GROUP"] = COSMOS3_UV_GROUP
os.environ["COSMOS3_TRANSFER_OUTPUT_ROOT"] = str(COSMOS3_TRANSFER_OUTPUT_ROOT)
os.environ["COSMOS3_MODEL"] = COSMOS3_MODEL
os.environ["COSMOS3_NUM_GPUS"] = COSMOS3_NUM_GPUS


def default_cache_path(name: str) -> str:
    root = os.environ.get("COSMOS3_CACHE_ROOT")
    if root:
        return str((Path(root).expanduser() / name).resolve())
    return str((COSMOS3_TRANSFER_ROOT / ".cache" / name).resolve())


os.environ["UV_CACHE_DIR"] = os.environ.get("COSMOS3_UV_CACHE_DIR", default_cache_path("uv"))
os.environ["HF_HOME"] = os.environ.get("COSMOS3_HF_HOME", default_cache_path("huggingface"))
# NGC PyTorch images: clear bundled libtorch from LD_LIBRARY_PATH before inference.
os.environ.pop("LD_LIBRARY_PATH", None)
# For Super, auto-expand CUDA_VISIBLE_DEVICES to cover COSMOS3_NUM_GPUS GPUs
# when the caller has not set it explicitly (default single-GPU '0' is not enough).
_cuda_default = (
    ",".join(str(i) for i in range(int(COSMOS3_NUM_GPUS)))
    if COSMOS3_MODEL == "Cosmos3-Super"
    else "0"
)
os.environ.setdefault("CUDA_VISIBLE_DEVICES", _cuda_default)
os.environ.setdefault("COSMOS3_MASTER_ADDR", "127.0.0.1")
os.environ.setdefault("COSMOS3_MASTER_PORT", free_local_port())

print("cosmos root:", COSMOS_ROOT)
print("transfer cookbook:", COSMOS3_TRANSFER_ROOT)
print("framework:", COSMOS3_REPO)
print("controls:", ", ".join(TRANSFER_CONTROLS))
print("output root:", COSMOS3_TRANSFER_OUTPUT_ROOT)
print("model:", os.environ["COSMOS3_MODEL"])
print("UV_CACHE_DIR:", os.environ["UV_CACHE_DIR"])
print("HF_HOME:", os.environ["HF_HOME"])
print("COSMOS3_UV_GROUP:", os.environ["COSMOS3_UV_GROUP"])
print("COSMOS3_NUM_GPUS:", os.environ["COSMOS3_NUM_GPUS"])
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])


cosmos root: /lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/users/trungp/repos/cosmos
transfer cookbook: /lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/users/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer
framework: /lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/users/trungp/repos/cosmos/packages/cosmos3
controls: edge, blur, depth, seg, wsm
output root: /lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/users/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer/outputs/notebooks
checkpoint: Cosmos3-Nano
UV_CACHE_DIR: /lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/users/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer/.cache/uv
HF_HOME: /lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/users/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer/.cache/huggingface
COSMOS3_UV_GROUP: cu130-train
CUDA_VISIBLE_DEVICES: 0,1,2,3


In [2]:
%%bash
set -euo pipefail
echo "hostname: $(hostname)"
echo "CUDA_VISIBLE_DEVICES=${CUDA_VISIBLE_DEVICES:-<unset>}"
if ! command -v nvidia-smi >/dev/null 2>&1; then
  echo "ERROR: nvidia-smi not found. Run on a GPU host (see §1)."
  exit 1
fi
nvidia-smi -L
GPU_COUNT=$(nvidia-smi -L 2>/dev/null | wc -l)
if [ "${GPU_COUNT:-0}" -lt 1 ]; then
  echo "ERROR: no GPUs visible. Allocate a GPU or set CUDA_VISIBLE_DEVICES."
  exit 1
fi
echo "OK: ${GPU_COUNT} GPU(s) visible on $(hostname)"

hostname: nvl72D130-T03
CUDA_VISIBLE_DEVICES=0,1,2,3


GPU 0: NVIDIA GB200 (UUID: GPU-76f8de40-8e91-e8e0-a3bc-eeb10770c592)
GPU 1: NVIDIA GB200 (UUID: GPU-

02153708-5e2c-816b-e8a2-e82617afaf3e)
GPU 2: NVIDIA GB200 (UUID: GPU-6be031fc-b525-6441-18f7-854d446

73407)
GPU 3: NVIDIA GB200 (UUID: GPU-3e3f578d-3e18-6667-0b90-8e708132b49b)


OK: 4 GPU(s) visible on nvl72D130-T03


## 4. Install system packages (Linux)

Framework guardrails and previews need **ffmpeg**, **git-lfs**, and graphics libraries (`libxcb1`, `libgl1`, …). On hosts with `apt-get` (NGC PyTorch container, many training images), run the next cell to install them.

If `apt-get` is unavailable, install the same packages with your OS package manager — see [Cosmos3 cookbooks README — System packages](../../README.md#system-packages-required-for-framework-guardrails).


In [3]:
%%bash
set -euo pipefail

PACKAGES=(curl ffmpeg git-lfs libgl1 libglib2.0-0 libx11-dev libxcb1 tree wget)

if command -v apt-get >/dev/null 2>&1; then
  export DEBIAN_FRONTEND=noninteractive
  echo "Installing system packages via apt-get..."
  apt-get update -qq
  apt-get install -y --no-install-recommends "${PACKAGES[@]}"
  echo "OK: apt packages installed"
else
  echo "NOTE: apt-get not found on this host."
  echo "If guardrails or OpenCV fail with libxcb.so.1, install manually:"
  echo "  ${PACKAGES[*]}"
  echo "See cookbooks/cosmos3/README.md (Framework guardrails)."
fi

for cmd in git ffmpeg; do
  command -v "$cmd" >/dev/null || { echo "ERROR: missing $cmd"; exit 1; }
done
if command -v git-lfs >/dev/null 2>&1; then
  git lfs install --skip-repo 2>/dev/null || true
  echo "git-lfs: OK"
else
  echo "WARN: git-lfs not in PATH (uv sync may still work with GIT_LFS_SKIP_SMUDGE=1)"
fi
echo "System package check complete."

Installing system packages via apt-get...


Reading package lists...

Building dependency tree...


Reading state information...

curl is already the newest version (8.5.0-2ubuntu10.9).
ffmpeg is already the newest version (7:6.1.

...

on (1.21.4-1ubuntu4.1).
0 upgraded, 0 newly installed, 0 to remove and 117 not upgraded.


OK: apt packages installed


Git LFS initialized.


git-lfs: OK


System package check complete.


## 5. Clone Cosmos Framework

Clones `COSMOS3_GIT_URL` into `COSMOS3_REPO` when the tree is not already present.

In [4]:
%%bash
set -euo pipefail

mkdir -p "$(dirname "$COSMOS3_REPO")"

if [ -d "$COSMOS3_REPO/.git" ]; then
  echo "Using existing framework checkout: $COSMOS3_REPO"
elif [ -f "$COSMOS3_REPO/pyproject.toml" ] && [ -d "$COSMOS3_REPO/cosmos_framework" ]; then
  echo "Using existing framework tree (no .git): $COSMOS3_REPO"
else
  echo "Cloning $COSMOS3_GIT_URL into $COSMOS3_REPO"
  git clone "$COSMOS3_GIT_URL" "$COSMOS3_REPO"
fi

cd "$COSMOS3_REPO"
if [ -d .git ]; then
  git status --short --branch
  git remote -v
fi


Using existing framework checkout: /lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/users

/trungp/repos/cosmos/packages/cosmos3


## feature/transfer-control-guidance...fork/feature/transfer-control-guidance
 M uv.lock


fork	https://github.com/trungtpham/cosmos-framework.git (fetch)
fork	https://github.com/trungtpham/c

osmos-framework.git (push)
origin	https://github.com/NVIDIA/cosmos-framework.git (fetch)
origin	http

s://github.com/NVIDIA/cosmos-framework.git (push)


## 6. Install Cosmos Framework Dependencies

Installs [`uv`](https://docs.astral.sh/uv/) if missing, then runs `uv sync` to create `packages/cosmos3/.venv`. Uses `COSMOS3_UV_GROUP` from §2.

**Skip:** if `.venv` already imports `cosmos_framework` with CUDA available, the next cell skips `uv sync` (fast re-runs). Set `COSMOS3_FORCE_UV_SYNC=1` to force a full re-sync.

For `jupyter execute` on a GPU node, set `COSMOS3_UV_CACHE_DIR` / `COSMOS3_HF_HOME` (or `COSMOS3_CACHE_ROOT`) in §2 first.

If you change `COSMOS3_UV_GROUP`, **re-run this cell** before inference.


In [5]:
%%bash
set -euo pipefail
unset LD_LIBRARY_PATH

if ! command -v uv >/dev/null 2>&1; then
  echo "Installing uv..."
  curl -LsSf https://astral.sh/uv/install.sh | sh
  # shellcheck disable=SC1091
  source "${HOME}/.local/bin/env" 2>/dev/null || export PATH="${HOME}/.local/bin:${PATH}"
fi
uv self update 2>/dev/null || true
uv --version

export GIT_LFS_SKIP_SMUDGE=1
mkdir -p "$UV_CACHE_DIR" "$HF_HOME"
cd "$COSMOS3_REPO"
export UV_CACHE_DIR="${UV_CACHE_DIR:?set paths in §2 (run the configure cell first)}"
export UV_PROJECT_ENVIRONMENT="${UV_PROJECT_ENVIRONMENT:-$COSMOS3_REPO/.venv}"
export UV_HTTP_TIMEOUT="${UV_HTTP_TIMEOUT:-600}"
echo "UV_CACHE_DIR=$UV_CACHE_DIR"
echo "COSMOS3_UV_GROUP=$COSMOS3_UV_GROUP"
echo "UV_HTTP_TIMEOUT=$UV_HTTP_TIMEOUT"

if [ -z "${COSMOS3_FORCE_UV_SYNC:-}" ] && [ -x ".venv/bin/python" ]; then
  if env -u LD_LIBRARY_PATH .venv/bin/python -c \
      'import cosmos_framework, torch; assert torch.cuda.is_available()'; then
    echo "venv ready at $COSMOS3_REPO/.venv — skipping uv sync (set COSMOS3_FORCE_UV_SYNC=1 to re-sync)"
    uv pip install imageio imageio-ffmpeg
    env -u LD_LIBRARY_PATH .venv/bin/python -c \
      'import cosmos_framework, torch; print("venv OK (skipped sync)")'
    exit 0
  fi
  echo "Existing .venv failed CUDA/framework check — running full uv sync..."
fi

attempt=1
max_attempts=3
until uv sync --all-extras --group="$COSMOS3_UV_GROUP"; do
  if [ "$attempt" -ge "$max_attempts" ]; then
    echo "ERROR: uv sync failed after $max_attempts attempts (PyPI timeout or network)."
    exit 1
  fi
  echo "uv sync attempt $attempt failed; retrying in 30s..."
  attempt=$((attempt + 1))
  sleep 30
done

uv pip install imageio imageio-ffmpeg

env -u LD_LIBRARY_PATH .venv/bin/python -c \
  'import cosmos_framework, torch; assert torch.cuda.is_available(); print("venv OK")'
echo "Install complete: $COSMOS3_REPO/.venv"


uv 0.8.17


UV_CACHE_DIR=/lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/users/trungp/repos/cosmos/c

ookbooks/cosmos3/generator/transfer/.cache/uv


COSMOS3_UV_GROUP=cu130-train


UV_HTTP_TIMEOUT=600


...

t-dependencies`, `build-constraint-dependencies`, `environments`, `required-environments`, `conflict

s`, `workspace`, `sources`, `managed`, `package`, `default-groups`, `dependency-groups`, `dev-depend

encies`, `build-backend`



Audited 2 packages in 218ms


venv OK (skipped sync)


## 6.5. Authenticate with Hugging Face

Downloads Cosmos3 weights from Hugging Face during first inference. This cell uses `HF_TOKEN` from §2 if set, or falls back to an existing `hf login` session.


In [ ]:
%%bash
set -euo pipefail

HF_BIN="$COSMOS3_REPO/.venv/bin/huggingface-cli"
[ -x "$HF_BIN" ] || HF_BIN=$(command -v huggingface-cli 2>/dev/null || echo "")

if [ -n "${HF_TOKEN:-}" ]; then
  echo "Logging in with HF_TOKEN..."
  if [ -n "$HF_BIN" ]; then
    "$HF_BIN" login --token "$HF_TOKEN" --add-to-git-credential 2>/dev/null || true
  fi
  echo "HF_TOKEN: set"
else
  echo "HF_TOKEN not set — checking for existing login session..."
  if [ -n "$HF_BIN" ] && "$HF_BIN" whoami >/dev/null 2>&1; then
    echo "Already logged in as: $(\"$HF_BIN\" whoami 2>/dev/null | head -1)"
  else
    echo "WARNING: No HF_TOKEN and no active login session."
    echo "Inference will fail when downloading weights. Fix by either:"
    echo "  1. Setting HF_TOKEN in §2 and re-running from the top, or"
    echo "  2. Running: uvx hf@latest auth login"
  fi
fi


## 7. Verify GPU Environment


In [6]:
import subprocess

verify_code = r'''
import sys
import torch
print("uv group (env):", __import__("os").environ.get("COSMOS3_UV_GROUP", "?"))
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device 0:", torch.cuda.get_device_name(0))
else:
    print("FIX: set COSMOS3_UV_GROUP in §2 (cu130-train or cu128-train), re-run §6 install, then this cell.")
    sys.exit(1)
'''
result = subprocess.run(
    [str(COSMOS3_REPO / ".venv" / "bin" / "python"), "-c", verify_code],
    cwd=str(COSMOS3_REPO),
    env=os.environ.copy(),
)
if result.returncode != 0:
    raise RuntimeError(
        "CUDA not available. Pass §3 first, then re-run §6 install with the correct COSMOS3_UV_GROUP."
    )


uv group (env): cu130-train
torch: 2.10.0+cu130
torch cuda: 13.0
cuda available: True
device count: 4
device 0: NVIDIA GB200


## 8. Input Specs and Preview Helpers

Checked-in [`specs/<control>.json`](./specs) are model-agnostic — the same spec runs with Nano or Super. Model selection is controlled entirely by `COSMOS3_MODEL` set in §2.

Inference (§9–§13) writes videos to:

```text
<COSMOS3_TRANSFER_OUTPUT_ROOT>/<model>/transfer_<control>/vision.mp4
```

For example:

```text
outputs/notebooks/Cosmos3-Nano/transfer_edge/vision.mp4
outputs/notebooks/Cosmos3-Super/transfer_edge/vision.mp4
```


In [7]:
missing = [c for c in TRANSFER_CONTROLS if not (COSMOS3_SPECS_DIR / f"{c}.json").is_file()]
if missing:
    raise FileNotFoundError(f"missing checked-in specs for {missing} under {COSMOS3_SPECS_DIR}")
print("Using specs:", ", ".join(f"{c}.json" for c in TRANSFER_CONTROLS))


Using specs: edge.json, blur.json, depth.json, seg.json, wsm.json


In [8]:
from preview_helpers import load_transfer_spec, resolve_spec_path

for control in TRANSFER_CONTROLS:
    spec = load_transfer_spec(control)
    block = spec[control]
    print(
        f"{control}: frames={spec.get('num_frames')} fps={spec.get('fps')} "
        f"guidance={spec.get('guidance')} control_guidance={spec.get('control_guidance')} "
        f"control={resolve_spec_path(block['control_path'])}"
    )


edge: frames=121 fps=30 guidance=3.0 control_guidance=1.5 control=/lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/users/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer/assets/edge/control_edge.mp4
blur: frames=121 fps=30 guidance=3.0 control_guidance=1.5 control=/lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/users/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer/assets/blur/control_blur.mp4
depth: frames=121 fps=30 guidance=3.0 control_guidance=1.5 control=/lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/users/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer/assets/depth/control_depth.mp4
seg: frames=121 fps=30 guidance=3.0 control_guidance=2.0 control=/lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/users/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer/assets/seg/control_seg.mp4
wsm: frames=101 fps=10 guidance=1.0 control_guidance=3.0 control=/lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/use

## 9. Edge (Canny) Transfer

Run after §7 reports `cuda available: True`.

Precomputed edge control (`control_edge.mp4`) + caption. Output:

```text
<COSMOS3_TRANSFER_OUTPUT_ROOT>/edge/transfer_edge/vision.mp4
```


In [9]:
%%bash
set -euo pipefail
unset LD_LIBRARY_PATH
CONTROL=edge
MODEL="${COSMOS3_MODEL:-Cosmos3-Nano}"
SPEC="$COSMOS3_TRANSFER_ROOT/specs/${CONTROL}.json"
OUT_DIR="$COSMOS3_TRANSFER_OUTPUT_ROOT/${MODEL}"
mkdir -p "$OUT_DIR"
echo "control=$CONTROL model=$MODEL spec=$SPEC output=$OUT_DIR"
cd "$COSMOS3_REPO"
if [ "$MODEL" = "Cosmos3-Super" ]; then
  CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES}" \
  .venv/bin/torchrun \
    --nproc-per-node="${COSMOS3_NUM_GPUS}" \
    --master-addr="${COSMOS3_MASTER_ADDR}" \
    --master-port="${COSMOS3_MASTER_PORT}" \
    -m cosmos_framework.scripts.inference \
    --parallelism-preset=throughput \
    -i "$SPEC" \
    -o "$OUT_DIR" \
    --checkpoint-path "$MODEL" \
    --seed 2025
else
  CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES}" \
  .venv/bin/python -m cosmos_framework.scripts.inference \
    --parallelism-preset=latency \
    -i "$SPEC" \
    -o "$OUT_DIR" \
    --checkpoint-path "$MODEL" \
    --seed 2025
fi


control=edge spec=/lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/users/trungp/repos/cos

mos/cookbooks/cosmos3/generator/transfer/specs/edge.json output=/lustre/fsw/portfolios/cosmos/projec

ts/cosmos_base_training/users/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer/outputs/noteb

ooks/edge checkpoint=Cosmos3-Nano


[06-08 21:28:13|job=|INFO|cosmos_framework/inference/common/init.py:127:_init_log_files] Console log

...

ge.mp4'


[06-08 21:33:49|job=|SUCCESS|cosmos_framework/inference/inference.py:1634:_generate_transfer_batch] 

[RANK 0] Saved transfer outputs to '/lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/user

s/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer/outputs/notebooks/edge/transfer_edge/samp

le_outputs.json'


### Preview edge


In [10]:
import os
import sys
from pathlib import Path

_root = Path.cwd()
if not (_root / "preview_helpers.py").is_file():
    for p in [_root, *_root.parents]:
        cand = p / "cookbooks" / "cosmos3" / "generator" / "transfer"
        if (cand / "preview_helpers.py").is_file():
            _root = cand
            break
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from preview_helpers import preview_transfer

preview_transfer("edge")


edge control: control_edge.mp4 (678 KB -> 570 KB preview)


edge generated: vision.mp4 (25660 KB -> 187 KB preview)


## 10. Blur Transfer

Blurred-reference control (`control_blur.mp4`) + caption. Output: `.../blur/transfer_blur/vision.mp4`.


In [11]:
%%bash
set -euo pipefail

CONTROL=blur
MODEL="${COSMOS3_MODEL:-Cosmos3-Nano}"
SPEC="$COSMOS3_TRANSFER_ROOT/specs/${CONTROL}.json"
OUT_DIR="$COSMOS3_TRANSFER_OUTPUT_ROOT/${MODEL}"
mkdir -p "$OUT_DIR"
echo "control=$CONTROL model=$MODEL spec=$SPEC output=$OUT_DIR"
cd "$COSMOS3_REPO"
if [ "$MODEL" = "Cosmos3-Super" ]; then
  CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES}" \
  .venv/bin/torchrun \
    --nproc-per-node="${COSMOS3_NUM_GPUS}" \
    --master-addr="${COSMOS3_MASTER_ADDR}" \
    --master-port="${COSMOS3_MASTER_PORT}" \
    -m cosmos_framework.scripts.inference \
    --parallelism-preset=throughput \
    -i "$SPEC" \
    -o "$OUT_DIR" \
    --checkpoint-path "$MODEL" \
    --seed 2025
else
  CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES}" \
  .venv/bin/python -m cosmos_framework.scripts.inference \
    --parallelism-preset=latency \
    -i "$SPEC" \
    -o "$OUT_DIR" \
    --checkpoint-path "$MODEL" \
    --seed 2025
fi


control=blur spec=/lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/users/trungp/repos/cos

mos/cookbooks/cosmos3/generator/transfer/specs/blur.json output=/lustre/fsw/portfolios/cosmos/projec

ts/cosmos_base_training/users/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer/outputs/noteb

ooks/blur checkpoint=Cosmos3-Nano


[06-08 21:34:08|job=|INFO|cosmos_framework/inference/common/init.py:127:_init_log_files] Console log

...

ur.mp4'


[06-08 21:39:00|job=|SUCCESS|cosmos_framework/inference/inference.py:1634:_generate_transfer_batch] 

[RANK 0] Saved transfer outputs to '/lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/user

s/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer/outputs/notebooks/blur/transfer_blur/samp

le_outputs.json'


### Preview blur


In [12]:
import os
import sys
from pathlib import Path

_root = Path.cwd()
if not (_root / "preview_helpers.py").is_file():
    for p in [_root, *_root.parents]:
        cand = p / "cookbooks" / "cosmos3" / "generator" / "transfer"
        if (cand / "preview_helpers.py").is_file():
            _root = cand
            break
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from preview_helpers import preview_transfer

preview_transfer("blur")


blur control: control_blur.mp4 (399 KB -> 180 KB preview)


blur generated: vision.mp4 (27315 KB -> 220 KB preview)


## 11. Depth Transfer

Depth-map control (`control_depth.mp4`) + caption. Output: `.../depth/transfer_depth/vision.mp4`.


In [13]:
%%bash
set -euo pipefail
unset LD_LIBRARY_PATH
CONTROL=depth
MODEL="${COSMOS3_MODEL:-Cosmos3-Nano}"
SPEC="$COSMOS3_TRANSFER_ROOT/specs/${CONTROL}.json"
OUT_DIR="$COSMOS3_TRANSFER_OUTPUT_ROOT/${MODEL}"
mkdir -p "$OUT_DIR"
echo "control=$CONTROL model=$MODEL spec=$SPEC output=$OUT_DIR"
cd "$COSMOS3_REPO"
if [ "$MODEL" = "Cosmos3-Super" ]; then
  CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES}" \
  .venv/bin/torchrun \
    --nproc-per-node="${COSMOS3_NUM_GPUS}" \
    --master-addr="${COSMOS3_MASTER_ADDR}" \
    --master-port="${COSMOS3_MASTER_PORT}" \
    -m cosmos_framework.scripts.inference \
    --parallelism-preset=throughput \
    -i "$SPEC" \
    -o "$OUT_DIR" \
    --checkpoint-path "$MODEL" \
    --seed 2025
else
  CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES}" \
  .venv/bin/python -m cosmos_framework.scripts.inference \
    --parallelism-preset=latency \
    -i "$SPEC" \
    -o "$OUT_DIR" \
    --checkpoint-path "$MODEL" \
    --seed 2025
fi


control=depth spec=/lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/users/trungp/repos/co

smos/cookbooks/cosmos3/generator/transfer/specs/depth.json output=/lustre/fsw/portfolios/cosmos/proj

ects/cosmos_base_training/users/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer/outputs/not

ebooks/depth checkpoint=Cosmos3-Nano


[06-08 21:42:00|job=|INFO|cosmos_framework/inference/common/init.py:127:_init_log_files] Console log

...

depth.mp4'


[06-08 21:47:13|job=|SUCCESS|cosmos_framework/inference/inference.py:1634:_generate_transfer_batch] 

[RANK 0] Saved transfer outputs to '/lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/user

s/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer/outputs/notebooks/depth/transfer_depth/sa

mple_outputs.json'


### Preview depth


In [14]:
import os
import sys
from pathlib import Path

_root = Path.cwd()
if not (_root / "preview_helpers.py").is_file():
    for p in [_root, *_root.parents]:
        cand = p / "cookbooks" / "cosmos3" / "generator" / "transfer"
        if (cand / "preview_helpers.py").is_file():
            _root = cand
            break
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from preview_helpers import preview_transfer

preview_transfer("depth")


depth control: control_depth.mp4 (1027 KB -> 236 KB preview)


depth generated: vision.mp4 (47734 KB -> 1178 KB preview)


## 12. Segmentation Transfer

Segmentation-map control (`control_seg.mp4`) + caption. Output: `.../seg/transfer_seg/vision.mp4`.


In [15]:
%%bash
set -euo pipefail

CONTROL=seg
MODEL="${COSMOS3_MODEL:-Cosmos3-Nano}"
SPEC="$COSMOS3_TRANSFER_ROOT/specs/${CONTROL}.json"
OUT_DIR="$COSMOS3_TRANSFER_OUTPUT_ROOT/${MODEL}"
mkdir -p "$OUT_DIR"
echo "control=$CONTROL model=$MODEL spec=$SPEC output=$OUT_DIR"
cd "$COSMOS3_REPO"
if [ "$MODEL" = "Cosmos3-Super" ]; then
  CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES}" \
  .venv/bin/torchrun \
    --nproc-per-node="${COSMOS3_NUM_GPUS}" \
    --master-addr="${COSMOS3_MASTER_ADDR}" \
    --master-port="${COSMOS3_MASTER_PORT}" \
    -m cosmos_framework.scripts.inference \
    --parallelism-preset=throughput \
    -i "$SPEC" \
    -o "$OUT_DIR" \
    --checkpoint-path "$MODEL" \
    --seed 2025
else
  CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES}" \
  .venv/bin/python -m cosmos_framework.scripts.inference \
    --parallelism-preset=latency \
    -i "$SPEC" \
    -o "$OUT_DIR" \
    --checkpoint-path "$MODEL" \
    --seed 2025
fi


control=seg spec=/lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/users/trungp/repos/cosm

os/cookbooks/cosmos3/generator/transfer/specs/seg.json output=/lustre/fsw/portfolios/cosmos/projects

/cosmos_base_training/users/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer/outputs/noteboo

ks/seg checkpoint=Cosmos3-Nano


[06-08 21:47:33|job=|INFO|cosmos_framework/inference/common/init.py:127:_init_log_files] Console log

...

mp4'


[06-08 21:52:47|job=|SUCCESS|cosmos_framework/inference/inference.py:1634:_generate_transfer_batch] 

[RANK 0] Saved transfer outputs to '/lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/user

s/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer/outputs/notebooks/seg/transfer_seg/sample

_outputs.json'


### Preview seg


In [16]:
import os
import sys
from pathlib import Path

_root = Path.cwd()
if not (_root / "preview_helpers.py").is_file():
    for p in [_root, *_root.parents]:
        cand = p / "cookbooks" / "cosmos3" / "generator" / "transfer"
        if (cand / "preview_helpers.py").is_file():
            _root = cand
            break
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from preview_helpers import preview_transfer

preview_transfer("seg")


seg control: control_seg.mp4 (4883 KB -> 425 KB preview)


seg generated: vision.mp4 (31289 KB -> 207 KB preview)


## 13. World Scenario (WSM) Transfer

World-scenario control (`control_wsm.mp4`) + caption. Output: `.../wsm/transfer_wsm/vision.mp4`.


In [17]:
%%bash
set -euo pipefail
unset LD_LIBRARY_PATH
CONTROL=wsm
MODEL="${COSMOS3_MODEL:-Cosmos3-Nano}"
SPEC="$COSMOS3_TRANSFER_ROOT/specs/${CONTROL}.json"
OUT_DIR="$COSMOS3_TRANSFER_OUTPUT_ROOT/${MODEL}"
mkdir -p "$OUT_DIR"
echo "control=$CONTROL model=$MODEL spec=$SPEC output=$OUT_DIR"
cd "$COSMOS3_REPO"
if [ "$MODEL" = "Cosmos3-Super" ]; then
  CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES}" \
  .venv/bin/torchrun \
    --nproc-per-node="${COSMOS3_NUM_GPUS}" \
    --master-addr="${COSMOS3_MASTER_ADDR}" \
    --master-port="${COSMOS3_MASTER_PORT}" \
    -m cosmos_framework.scripts.inference \
    --parallelism-preset=throughput \
    -i "$SPEC" \
    -o "$OUT_DIR" \
    --checkpoint-path "$MODEL" \
    --seed 2025
else
  CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES}" \
  .venv/bin/python -m cosmos_framework.scripts.inference \
    --parallelism-preset=latency \
    -i "$SPEC" \
    -o "$OUT_DIR" \
    --checkpoint-path "$MODEL" \
    --seed 2025
fi


control=wsm spec=/lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/users/trungp/repos/cosm

os/cookbooks/cosmos3/generator/transfer/specs/wsm.json output=/lustre/fsw/portfolios/cosmos/projects

/cosmos_base_training/users/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer/outputs/noteboo

ks/wsm checkpoint=Cosmos3-Nano


[06-08 21:53:06|job=|INFO|cosmos_framework/inference/common/init.py:127:_init_log_files] Console log

...

mp4'


[06-08 21:55:37|job=|SUCCESS|cosmos_framework/inference/inference.py:1634:_generate_transfer_batch] 

[RANK 0] Saved transfer outputs to '/lustre/fsw/portfolios/cosmos/projects/cosmos_base_training/user

s/trungp/repos/cosmos/cookbooks/cosmos3/generator/transfer/outputs/notebooks/wsm/transfer_wsm/sample

_outputs.json'


### Preview wsm


In [18]:
import os
import sys
from pathlib import Path

_root = Path.cwd()
if not (_root / "preview_helpers.py").is_file():
    for p in [_root, *_root.parents]:
        cand = p / "cookbooks" / "cosmos3" / "generator" / "transfer"
        if (cand / "preview_helpers.py").is_file():
            _root = cand
            break
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from preview_helpers import preview_transfer

preview_transfer("wsm")


wsm control: control_wsm.mp4 (3527 KB -> 191 KB preview)


wsm generated: vision.mp4 (23384 KB -> 387 KB preview)
